<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/visualize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Data Visualization

Load pre-computed pipeline outputs (depth, extrinsics, tracks) from GCS and visualize them.

**No computation needed** — this notebook only loads and visualizes.

| Stage | What you'll see |
|-------|----------------|
| 1. Depth | Stereo disparity video, depth maps |
| 2. Extrinsics | Camera axes overlay, robot segmentation, fused 3D point cloud, 4D orbit |
| 3. Tracks v2 | Track statistics, per-camera 2D tracking video, 3D visualization, depth consistency |

---
## 0. Environment Setup

In [ ]:
# @title 0a. Clone repo & install dependencies
import os

REPO_DIR = "/content/droid"
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/yangyi02/droid.git {REPO_DIR}
else:
    print(f"⏭️ Repo already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive

%cd {REPO_DIR}
!bash setup.sh

In [ ]:
# @title 0b. Python imports & sys.path setup
import sys, os, json, random
import numpy as np
import torch
import mediapy as media

REPO_DIR = "/content/droid"
for p in [
    REPO_DIR,
    os.path.join(REPO_DIR, "third_party/s2m2/src"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(REPO_DIR)
os.environ['PYOPENGL_PLATFORM'] = 'egl'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

In [ ]:
# @title 0c. Download DROID metadata

import urllib.request

root_path = "/content/droid_raw/1.0.1"
base_url = "https://huggingface.co/KarlP/droid/resolve/main"
files = ["camera_serials.json", "episode_id_to_path.json",
         "keep_ranges_1_0_1.json", "cam2base_extrinsic_superset.json"]

os.makedirs(root_path, exist_ok=True)

for f in files:
    target_file = os.path.join(root_path, f)
    if not os.path.exists(target_file):
        print(f"Downloading: {f} ...")
        req = urllib.request.Request(f"{base_url}/{f}", headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req) as response, open(target_file, 'wb') as out_file:
                out_file.write(response.read())
        except Exception as e:
            print(f"❌ {f} Download failed: {e}")

def load_json(name):
    with open(os.path.join(root_path, name)) as f:
        return json.load(f)

serials_db = load_json(files[0])
id_to_path = load_json(files[1])
keep_ranges = load_json(files[2])
extrinsics_db = load_json(files[3])

with open("episodes_success.txt") as f:
    valid_ids = sorted([line.strip() for line in f if line.strip()])
print(f"✅ Metadata ready: {len(valid_ids)} episodes")

In [ ]:
# @title 0d. Select episode

# Option 1: Random
episode_id = random.choice(valid_ids)

# Option 2: Manual override (uncomment)
# episode_id = "ILIAD+5e938e3b+2023-07-20-11h-50m-51s"

print(f"🎯 Episode: {episode_id}")

In [ ]:
# @title 0e. Initialize scene_constants (lightweight, no SVO)
from compute_depth import init_episode

scene_constants = init_episode(
    episode_id,
    os.path.expanduser("~/droid_data/input/robotics/droid_raw/1.0.1"),
    id_to_path, serials_db, keep_ranges)
print(f"✅ scene_constants initialized: {list(scene_constants['camera'].keys())}")

---
## 1. Load & Visualize Depth

In [ ]:
# @title 1a. ☁️ Load depth from GCS

GCS_DEPTH = "gs://dm-tapnet/mv-tap/droid/depth"
local_cache = f"/content/droid_depth_cache/{episode_id}"
os.makedirs(local_cache, exist_ok=True)

# Download robot data
os.system(f"gsutil cp '{GCS_DEPTH}/{episode_id}/robot.npz' '{local_cache}/' > /dev/null 2>&1")
robot_data = np.load(f"{local_cache}/robot.npz", allow_pickle=True)
for k in ['joint_positions', 'gripper_positions', 'T_cam_ee_init', 'T_ee_base_all']:
    if k in robot_data:
        scene_constants['robot'][k] = robot_data[k]
if 'valid_indices' in robot_data:
    scene_constants['meta']['valid_indices'] = robot_data['valid_indices']
if 'wrist_serial' in robot_data:
    scene_constants['meta']['wrist_serial'] = str(robot_data['wrist_serial'].item())
wrist_serial = scene_constants['meta'].get('wrist_serial')
print(f"  ✅ robot.npz loaded")

# Download per-camera data
base_files = ["video_left.mp4", "video_right.mp4",
              "video_left_raw.mp4", "video_right_raw.mp4",
              "raw_depth.npz", "calibration.npz"]

for cam in scene_constants['camera']:
    cam_dir = os.path.join(local_cache, cam)
    os.makedirs(cam_dir, exist_ok=True)

    cam_files = list(base_files)
    if cam == wrist_serial:
        cam_files.extend(["original_raw_depth.npz", "gripper_mask.npz", "gripper_depth.npz"])

    gcs_files = " ".join([f"'{GCS_DEPTH}/{episode_id}/{cam}/{f}'" for f in cam_files])
    os.system(f"gsutil -m cp {gcs_files} '{cam_dir}/' > /dev/null 2>&1")

    # Videos
    for mem_key, fname in [("video_rgb", "video_left.mp4"), ("video_right", "video_right.mp4"),
                           ("video_raw_rgb", "video_left_raw.mp4"), ("video_raw_right", "video_right_raw.mp4")]:
        vid_path = os.path.join(cam_dir, fname)
        if os.path.exists(vid_path):
            scene_constants['camera'][cam][mem_key] = media.read_video(vid_path)

    # Depth
    depth_path = os.path.join(cam_dir, "raw_depth.npz")
    if os.path.exists(depth_path):
        scene_constants['camera'][cam]['raw_depth'] = np.load(depth_path)['depth'].astype(np.float32) / 1000.0

    # Wrist extras
    for npz_key, mem_key, is_depth in [
        ("original_raw_depth.npz", "original_raw_depth", True),
        ("gripper_mask.npz", "sam_real_masks", False),
        ("gripper_depth.npz", "empirical_gripper_depth", True)]:
        p = os.path.join(cam_dir, npz_key)
        if os.path.exists(p):
            d = np.load(p)
            key = 'depth' if 'depth' in d else 'mask'
            val = d[key]
            if is_depth:
                val = val.astype(np.float32) / 1000.0
            scene_constants['camera'][cam][mem_key] = val

    # Calibration
    calib_path = os.path.join(cam_dir, "calibration.npz")
    if os.path.exists(calib_path):
        c = np.load(calib_path)
        scene_constants['camera'][cam]['K_mat'] = c['K_calib_left']
        scene_constants['camera'][cam]['baseline'] = float(c['baseline'])
        scene_constants['camera'][cam]['zed_calibration'] = {
            'calibrated': {'K': c['K_calib_left'], 'disto': c['disto_calib_left'],
                           'K_right': c['K_calib_right'], 'disto_right': c['disto_calib_right']},
            'raw': {'K': c['K_raw_left'], 'disto': c['disto_raw_left'],
                    'K_right': c['K_raw_right'], 'disto_right': c['disto_raw_right']},
        }
    print(f"  ✅ Camera {cam} loaded")

print(f"✅ Depth LOADED from GCS")

In [ ]:
# @title 1b. 🎬 Visualize depth results
from core.visualization import inspect_dict_structure, render_multicam_disparity_video

inspect_dict_structure(scene_constants)

frames = render_multicam_disparity_video(scene_constants, max_frames=30)
media.show_video(frames, fps=10, title="Depth [left | right | disparity] per camera")

---
## 2. Load & Visualize Extrinsics

In [ ]:
# @title 2a. ☁️ Load extrinsics from GCS

GCS_EXT = "gs://dm-tapnet/mv-tap/droid/extrinsics"
local_ext_cache = f"/content/droid_extrinsics_cache/{episode_id}"
os.makedirs(local_ext_cache, exist_ok=True)

scene_state = {}
for cam in scene_constants['camera']:
    cam_dir = os.path.join(local_ext_cache, cam)
    os.makedirs(cam_dir, exist_ok=True)

    gcs_path = f"{GCS_EXT}/{episode_id}/{cam}/extrinsics.json"
    local_path = os.path.join(cam_dir, "extrinsics.json")
    os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")

    if os.path.exists(local_path):
        with open(local_path) as f:
            ext_data = json.load(f)
        scene_state[cam] = {
            'base_extrinsic': np.array(ext_data['base_extrinsic'], dtype=np.float32),
            'extrinsics': np.array(ext_data['extrinsics'], dtype=np.float32),
            'is_wrist': ext_data.get('is_wrist', False),
        }
        flag = "🦿" if scene_state[cam]['is_wrist'] else "🎥"
        print(f"  ✅ [{cam}] {flag} Shape: {scene_state[cam]['extrinsics'].shape}")
    else:
        print(f"  ⚠️ [{cam}] missing")

print("✅ Extrinsics LOADED from GCS")

In [ ]:
# @title 2b. 📐 Camera axes overlay
from core.visualization import render_cross_camera_axes

try:
    axes_frames = render_cross_camera_axes(scene_constants, scene_state, max_frames=30)
    if axes_frames:
        media.show_video(axes_frames, fps=10, title="Camera Axes Overlay")
except Exception as e:
    print(f"Axes visualization skipped: {e}")

In [ ]:
# @title 2c. 🤖 Robot segmentation video
from core.visualization import render_segmentation_video
from core.physics import PyBulletRenderer

if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

try:
    seg_frames = render_segmentation_video(scene_constants, scene_state, pb_renderer, max_frames=30)
    if seg_frames:
        media.show_video(seg_frames, fps=10, title="Robot Mask Overlay")
except Exception as e:
    print(f"Segmentation visualization skipped: {e}")

In [ ]:
# @title 2d. 🌐 Fused 3D point cloud
from core.visualization import render_fused_point_cloud

try:
    render_fused_point_cloud(scene_constants, scene_state, frame_idx=0, height=600, width=1000)
except Exception as e:
    print(f"Fused point cloud skipped: {e}")

In [ ]:
# @title 2e. 🎬 4D cinematic orbit
from core.visualization import render_cinematic_4d_orbit

try:
    orbit_frames = render_cinematic_4d_orbit(scene_constants, scene_state, max_frames=30)
    media.show_video(orbit_frames, fps=10, title="4D Orbit")
except Exception as e:
    print(f"4D Orbit skipped: {e}")

---
## 3. Load & Visualize Tracks v2

In [ ]:
# @title 3a. ☁️ Load tracks v2 from GCS

GCS_TRACKS2 = "gs://dm-tapnet/mv-tap/droid/tracks2"
local_tracks2_cache = f"/content/droid_tracks2_cache/{episode_id}"
os.makedirs(local_tracks2_cache, exist_ok=True)

# Download global 3D tracks + metadata
for fname in ["tracks_3d.npz", "track_metadata.npz"]:
    gcs_path = f"{GCS_TRACKS2}/{episode_id}/{fname}"
    local_path = os.path.join(local_tracks2_cache, fname)
    ret = os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")
    if ret == 0:
        print(f"  ✅ {fname}")
    else:
        print(f"  ⚠️  {fname} not found (skipping)")

# Load global tracks
data_3d = np.load(os.path.join(local_tracks2_cache, "tracks_3d.npz"))
final_traj_3d = data_3d["traj_3d"]       # (T, N, 3)
final_vis_global = data_3d["vis_global"]  # (T, N)

# Load static/robot split metadata
meta_path = os.path.join(local_tracks2_cache, "track_metadata.npz")
if os.path.exists(meta_path):
    meta = np.load(meta_path)
    n_static = int(meta["n_static"])
    n_robot = int(meta["n_robot"])
else:
    n_static, n_robot = final_traj_3d.shape[1], 0

T, N, _ = final_traj_3d.shape
print(f"  ✅ tracks_3d: {T} frames × {N} points ({n_static} static + {n_robot} robot)")

# Download per-camera 2D tracks
final_per_cam_tracks = {}
final_per_cam_vis = {}

for cam_id in scene_constants["camera"]:
    cam_cache = os.path.join(local_tracks2_cache, cam_id)
    os.makedirs(cam_cache, exist_ok=True)

    gcs_cam = f"{GCS_TRACKS2}/{episode_id}/{cam_id}"
    for fname in ["tracks_2d.npz", "intrinsics.npy", "extrinsics_w2c.npy"]:
        os.system(f"gsutil cp '{gcs_cam}/{fname}' '{cam_cache}/' > /dev/null 2>&1")

    t2d_path = os.path.join(cam_cache, "tracks_2d.npz")
    if os.path.exists(t2d_path):
        d = np.load(t2d_path)
        final_per_cam_tracks[cam_id] = d["traj_2d"]   # (T, N, 2)
        final_per_cam_vis[cam_id]    = d["vis_2d"]     # (T, N)
        print(f"  ✅ Camera [{cam_id}]: 2D tracks loaded")
    else:
        print(f"  ⚠️  Camera [{cam_id}]: tracks_2d.npz not found")

print(f"\n✅ Tracks v2 LOADED — {len(final_per_cam_tracks)} cameras, {n_static} static + {n_robot} robot = {N} points")

In [ ]:
# @title 3b. 📊 Track Summary Statistics
import numpy as np

T, N, _ = final_traj_3d.shape
camera_ids = list(scene_constants['camera'].keys())

print(f"Track Summary")
print(f"{'=' * 50}")
print(f"  Total points:    {N}  ({n_static} static + {n_robot} robot)")
print(f"  Total frames:    {T}")
print()

for cam_id in camera_ids:
    if cam_id not in final_per_cam_vis:
        continue
    vis = final_per_cam_vis[cam_id]
    vis_static = vis[:, :n_static]
    vis_robot = vis[:, n_static:]
    print(f"  📷 [{cam_id}]:")
    print(f"       Static visibility: {vis_static.mean()*100:.1f}% avg")
    print(f"       Robot  visibility: {vis_robot.mean()*100:.1f}% avg")
    print(f"       Total  visibility: {vis.mean()*100:.1f}% avg")

In [ ]:
# @title 3c. 🎬 Per-Camera 2D Tracking Video (static 🌈 + robot 🔴)
import importlib, cv2, numpy as np
import matplotlib.pyplot as plt
import core.visualization
importlib.reload(core.visualization)
from core.visualization import render_2d_tracking_video
import mediapy as media

camera_ids = list(scene_constants['camera'].keys())

# --- Precompute consistent colors ---
ref_cam = camera_ids[min(1, len(camera_ids) - 1)]
if n_static > 0:
    y_static = final_per_cam_tracks[ref_cam][0, :n_static, 1]
    norm_s = plt.Normalize(y_static.min(), y_static.max())
    static_colors = (plt.cm.gist_rainbow(norm_s(y_static))[:, :3] * 255).astype(np.uint8)
else:
    static_colors = np.zeros((0, 3), dtype=np.uint8)

robot_colors = np.full((n_robot, 3), [255, 50, 50], dtype=np.uint8) if n_robot > 0 else np.zeros((0, 3), dtype=np.uint8)
combined_colors = np.concatenate([static_colors, robot_colors], axis=0)

# --- Static-only tracks ---
all_frames_static = []
for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    tracks = final_per_cam_tracks[cam_id][:, :n_static, :]
    vis = final_per_cam_vis[cam_id][:, :n_static]
    frames = render_2d_tracking_video(
        cam_data['video_rgb'], tracks, vis,
        global_colors=static_colors,
        tgt_size=(256, 456), linewidth=1, max_frames=60)
    for f in frames:
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    all_frames_static.append(np.array(frames))

if all_frames_static:
    combined = np.concatenate(all_frames_static, axis=2)
    media.show_video(combined, fps=10,
                     title=f"Static Background Tracks ({n_static} points) — All Cameras")

# --- Robot-only tracks ---
if n_robot > 0:
    all_frames_robot = []
    for cam_id in camera_ids:
        cam_data = scene_constants['camera'][cam_id]
        tracks = final_per_cam_tracks[cam_id][:, n_static:, :]
        vis = final_per_cam_vis[cam_id][:, n_static:]
        frames = render_2d_tracking_video(
            cam_data['video_rgb'], tracks, vis,
            global_colors=robot_colors,
            tgt_size=(256, 456), linewidth=1, max_frames=60)
        for f in frames:
            cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
            cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        all_frames_robot.append(np.array(frames))

    if all_frames_robot:
        combined_robot = np.concatenate(all_frames_robot, axis=2)
        media.show_video(combined_robot, fps=10,
                         title=f"Robot Tracks ({n_robot} points) — All Cameras")

# --- Combined: static (rainbow) + robot (red) ---
all_frames_both = []
for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    frames = render_2d_tracking_video(
        cam_data['video_rgb'],
        final_per_cam_tracks[cam_id],
        final_per_cam_vis[cam_id],
        global_colors=combined_colors,
        tgt_size=(256, 456), linewidth=1, max_frames=60)
    for f in frames:
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    all_frames_both.append(np.array(frames))

if all_frames_both:
    combined_both = np.concatenate(all_frames_both, axis=2)
    media.show_video(combined_both, fps=10,
                     title=f"All Tracks ({n_static} static 🌈 + {n_robot} robot 🔴)")

In [ ]:
# @title 3d. 🌐 Static Points 3D Visualization (interactive Plotly)
import numpy as np
import plotly.graph_objects as go
from core.visualization import show_plotly_point_cloud
from core.geometry import unproject_to_3d

camera_ids = list(scene_constants['camera'].keys())

if n_static > 0:
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=static_pts_3d[:, 0], y=static_pts_3d[:, 1], z=static_pts_3d[:, 2],
        mode='markers',
        marker=dict(
            size=3,
            color=[f'rgb({r},{g},{b})' for r, g, b in static_rgb],
        ),
        name=f"Static Background ({n_static})",
    ))

    if n_robot > 0:
        robot_t0 = final_traj_3d[0, n_static:, :]
        fig.add_trace(go.Scatter3d(
            x=robot_t0[:, 0], y=robot_t0[:, 1], z=robot_t0[:, 2],
            mode='markers',
            marker=dict(size=3, color='red'),
            name=f"Robot ({n_robot})",
        ))

    # Add fused point cloud from one static camera for context
    t_ctx = 0
    cam_ctx = camera_ids[1]
    cam_data = scene_constants['camera'][cam_ctx]
    pts_ctx, rgb_ctx = unproject_to_3d(
        cam_data['raw_depth'][t_ctx], cam_data['video_rgb'][t_ctx],
        cam_data['K_mat'], scene_state[cam_ctx]['extrinsics'][t_ctx],
        max_depth=2.0)
    idx_ctx = np.random.permutation(len(pts_ctx))[:50000]
    fig.add_trace(go.Scatter3d(
        x=pts_ctx[idx_ctx, 0], y=pts_ctx[idx_ctx, 1], z=pts_ctx[idx_ctx, 2],
        mode='markers',
        marker=dict(
            size=1,
            color=[f'rgba({r},{g},{b},0.2)' for r, g, b in rgb_ctx[idx_ctx]],
        ),
        name=f"Context PCL [{cam_ctx[:8]}]",
        opacity=0.3,
    ))

    fig.update_layout(
        title=f"Track Points: {n_static} static + {n_robot} robot",
        scene=dict(aspectmode='data'),
        height=700, width=1000,
    )
    fig.show()
else:
    print("⚠️ No static points to visualize.")

In [ ]:
# @title 3e. 🎯 Point Depth Consistency Analysis (Static vs. Robot)
import numpy as np
import matplotlib.pyplot as plt
from core.geometry import project_points_np

camera_ids = list(scene_constants['camera'].keys())
T_frames = final_traj_3d.shape[0]

errors = {cam: {'static': [], 'robot': [], 'all': []} for cam in camera_ids}

def compute_depth_residual_mm(pts_3d, K, extrinsics, raw_depth, w_img, h_img):
    if len(pts_3d) == 0:
        return []
    u_proj, v_proj, z_proj = project_points_np(pts_3d, K, extrinsics)
    ui = np.clip(np.round(u_proj).astype(int), 0, w_img - 1)
    vi = np.clip(np.round(v_proj).astype(int), 0, h_img - 1)
    z_obs = raw_depth[vi, ui]
    valid = (z_obs > 0.05) & (z_proj > 0)
    return (np.abs(z_proj[valid] - z_obs[valid]) * 1000.0).tolist() if valid.any() else []

for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    K, (h_img, w_img) = cam_data['K_mat'], cam_data['video_rgb'][0].shape[:2]

    for t in range(T_frames):
        raw_depth = cam_data['raw_depth'][t]
        ext = scene_state[cam_id]['extrinsics'][t]
        vis_t = final_per_cam_vis[cam_id][t]

        if n_static > 0:
            errors[cam_id]['static'].extend(
                compute_depth_residual_mm(final_traj_3d[t, :n_static][vis_t[:n_static]], K, ext, raw_depth, w_img, h_img))
        if n_robot > 0:
            errors[cam_id]['robot'].extend(
                compute_depth_residual_mm(final_traj_3d[t, n_static:][vis_t[n_static:]], K, ext, raw_depth, w_img, h_img))
        errors[cam_id]['all'].extend(
            compute_depth_residual_mm(final_traj_3d[t, vis_t], K, ext, raw_depth, w_img, h_img))

# --- Visualization ---
fig, axes = plt.subplots(1, len(camera_ids), figsize=(4.8 * len(camera_ids), 3.8), sharey=True)
if len(camera_ids) == 1:
    axes = [axes]

palette = {'static': '#2da44e', 'robot': '#cf222e', 'all': '#0969da'}

for ax, cam_id in zip(axes, camera_ids):
    s_err = np.array(errors[cam_id]['static'])
    r_err = np.array(errors[cam_id]['robot'])
    a_err = np.array(errors[cam_id]['all'])

    if len(s_err):
        med_s = np.median(s_err)
        ax.hist(s_err, bins=40, range=(0, 40), alpha=0.4, color=palette['static'], label=f'Static (Med: {med_s:.1f} mm)')
        ax.axvline(med_s, color=palette['static'], linestyle='--', linewidth=1.2)

    if len(r_err):
        med_r = np.median(r_err)
        ax.hist(r_err, bins=40, range=(0, 40), alpha=0.4, color=palette['robot'], label=f'Robot (Med: {med_r:.1f} mm)')
        ax.axvline(med_r, color=palette['robot'], linestyle='--', linewidth=1.2)

    if len(a_err):
        med_a = np.median(a_err)
        ax.axvline(med_a, color=palette['all'], linestyle='-', linewidth=1.6, label=f'Overall (Med: {med_a:.1f} mm)')

    ax.set_title(f'Camera [{cam_id[:8]}]', fontsize=11, pad=10, fontweight='bold')
    ax.set_xlabel('Depth Residual Error (mm)', fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=True, facecolor='white', framealpha=0.95, fontsize=8)

axes[0].set_ylabel('Observation Count', fontsize=9)
plt.suptitle(f'Depth Consistency (Static={n_static}, Robot={n_robot}, Total={N})', fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# @title 3f. 🎬 4D Orbit Video with Tracks + Camera Frustums

import importlib, core.visualization
importlib.reload(core.visualization)
from core.visualization import render_4d_orbit_with_tracks
import mediapy as media

orbit_frames = render_4d_orbit_with_tracks(
    scene_constants, scene_state,
    tracks_3d=final_traj_3d,
    track_history=5,
    track_sphere_radius=0.006,
    frustum_depth=0.12,
    max_render_points=300000,
    max_render_tracks=500,
    width=960, height=540,
    orbit_center=(0.4, 0.0, 0.0),
    orbit_radius=1.2,
    camera_height=0.5,
    max_frames=60,
)
media.show_video(orbit_frames, fps=10,
                 title="4D Orbit — Point Cloud + Tracks + Cameras")